# 🚀 NexRouter v5.1.0 Apex Cloud Firmware Builder (Google Colab)
**Target Device:** Xiaomi Redmi Router AC2100 (`RM2100`, MediaTek `MT7621AT` + `MT7530` 1.2G TRGMII + `MT7615N` 4×4 5GHz + `MT7603EN` 2.4GHz + `2× Qorvo QPF4200` Wi-Fi 6 FEMs + `EIP-93` Crypto Engine)

This Google Colab notebook builds your custom **NexRouter v5.1.0** sysupgrade firmware image directly in the cloud with **Pre-Installed Customized LuCI Argon Theme (`v2.4.8` Dark Glassmorphic) + `luci-app-argon-config` + `luci-app-filemanager` + `/nex/` Telemetry SPA**:
- **Option 1 (`snapshot` — Default, ~3 mins):** Latest OpenWrt `snapshot` ImageBuilder (`--release snapshot`) with Customized Argon Theme, `luci-app-filemanager`, `kmod-crypto-hw-eip93`, `bridger` eBPF, 64 MiB ZRAM, Hardware WPA3-SAE (`H2E`), Wi-Fi 6 `MBO`/`RNR`/`OKC`, and live `nexrouter-accel` MMIO register programming (`0x4208`/`0x4210`/`0x21210`/`0x10f18`).
- **Option 2 (`24.10.2` — Stable LTS, ~3 mins):** OpenWrt `24.10.2` LTS ImageBuilder (`--release 24.10.2`) with the full NexRouter v5.1.0 stack, Customized Argon Theme, and `luci-app-filemanager`.
- **Option 3 (`full-source-snapshot` — ~45-60 mins):** Full OpenWrt source compile (`--branch main` or `v24.10.2`) injecting all **16 custom C Kernel (`100`–`106`) & `mt76` (`200`–`208`) patches** (`1.2 Gbps` TRGMII switch trunk, `-O3 -mtune=1004kc -mdsp`, `STA_REC_BF` 4×4 MU-MIMO PFMU, Wi-Fi 6 `802.3` HW TX encap, and WPDMA IRQ coalescing).

In [ ]:
# @title 1. Install Linux Build Dependencies & Node.js 20+
%%bash
set -e
sudo apt-get update -qq
sudo apt-get install -y -qq build-essential clang flex bison g++ gawk \
  gcc-multilib g++-multilib gettext git libncurses5-dev libssl-dev \
  python3-setuptools rsync swig unzip zlib1g-dev file wget curl \
  zstd xz-utils squashfs-tools jq gcc-mipsel-linux-gnu
if ! command -v node >/dev/null 2>&1; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
  sudo apt-get install -y -qq nodejs
fi
echo "Node version: $(node -v)"
echo "Dependencies ready!"

In [ ]:
# @title 2. Clone Your Git Repository (or Upload `redmi-ac2100-os-v2.zip`)
import os

GIT_REPO_URL = "" # @param {type:"string"} 
WORK_DIR = "/content/redmi-ac2100-os-v3"

if GIT_REPO_URL.strip():
    !rm -rf {WORK_DIR}
    !git clone {GIT_REPO_URL} {WORK_DIR}
elif os.path.exists("/content/redmi-ac2100-os-v2.zip"):
    !rm -rf {WORK_DIR}
    !unzip -q /content/redmi-ac2100-os-v2.zip -d /content/
elif os.path.exists("../build/build.sh"):
    WORK_DIR = os.path.abspath("..")
else:
    print("Upload redmi-ac2100-os-v2.zip to /content/ in the Colab sidebar or paste your GIT_REPO_URL above.")

if os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)
    print("Working directory:", os.getcwd())
    !ls -la

In [ ]:
# @title 3. Build Firmware Image (Default: Latest OpenWrt `snapshot`)
BUILD_TRACK = "snapshot" # @param ["snapshot", "24.10.2", "full-source-snapshot", "full-source-24.10.2"]

if BUILD_TRACK == "snapshot":
    !chmod +x build/build.sh && ./build/build.sh --release snapshot
elif BUILD_TRACK == "24.10.2":
    !chmod +x build/build.sh && ./build/build.sh --release 24.10.2
elif BUILD_TRACK == "full-source-snapshot":
    !chmod +x build/build-from-source.sh && ./build/build-from-source.sh --branch main
elif BUILD_TRACK == "full-source-24.10.2":
    !chmod +x build/build-from-source.sh && ./build/build-from-source.sh --branch v24.10.2

!node analysis/tools/fwscan.mjs out/*sysupgrade*.bin

In [ ]:
# @title 4. Download Built `*sysupgrade*.bin` Firmware to Your PC
import glob
from google.colab import files

bins = sorted(glob.glob("out/*sysupgrade*.bin"))
for b in bins:
    print("Downloading:", b)
    files.download(b)